# Vocabulary Building

In most natural language processing (NLP) tasks, the initial step in preparing your data is to extract a vocabulary of words from your corpus (i.e. input texts). You will need to define how to represent the texts into numeric features which can be used to train a neural network. Tensorflow and Keras makes it easy to generate these using its APIs. You will see how to do that in the next cells.

The code below takes a list of sentences, then takes each word in those sentences and assigns it to an integer. This is done using the TextVectorization() preprocessing layer and its adapt() method.

As mentioned in the docs above, this layer does several things including:

    1. Standardizing each example. The default behavior is to lowercase and strip punctuation. See its standardize argument for other options.
    2. Splitting each example into substrings. By default, it will split into words. See its split argument for other options.
    3. Recombining substrings into tokens. See its ngrams argument for reference.
    4. Indexing tokens.
    5. Transforming each example using this index, either into a vector of ints or a dense float vector.



### TensorFlow

In [1]:
import tensorflow as tf

# sample inputs

sentences = [
    "I love my dog",
    "i love my cat"
    ]

# Initialize the layeer
vectorized_layer = tf.keras.layers.TextVectorization()

# Build the vocab
vectorized_layer.adapt(sentences)

# get vocab
vocab = vectorized_layer.get_vocabulary(include_special_tokens = False)

print(vocab)

2026-01-15 19:14:19.476864: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-15 19:14:20.306537: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


['my', 'love', 'i', 'dog', 'cat']


2026-01-15 19:14:21.372932: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:996] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-01-15 19:14:21.378482: W tensorflow/core/common_runtime/gpu/gpu_device.cc:1956] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...



The resulting vocabulary will be a list where more frequently used words will have a lower index. By default, it will also reserve indices for special tokens but , for clarity, let's reserve that for later.

In [2]:
for index, word in enumerate(vocab):
    print(index, word)

0 my
1 love
2 i
3 dog
4 cat


### PyTorch

In [3]:
# !pip show torch torchtext


In [4]:
import torch
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer

sentences = [
    "I love my dog",
    "i love my cat"
]

# whitespace tokenizer
tokenizer = get_tokenizer("basic_english")

# define yield
def yield_token(sentences):
    for sentenece in sentences:
        yield tokenizer(sentenece)

vocab = build_vocab_from_iterator(yield_token(sentences), specials=[])
vocab_list = vocab.get_itos() # index to string
vocab_list_2 = vocab.get_stoi() # string to index
print(vocab)
print(vocab_list)
print(vocab_list_2)



Vocab()
['i', 'love', 'my', 'cat', 'dog']
{'dog': 4, 'cat': 3, 'my': 2, 'love': 1, 'i': 0}


/home/pushkar/ActionAI_2/nlp2llm/.venv-mix/lib/python3.11/site-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/pushkar/ActionAI_2/nlp2llm/.venv-mix/lib/python3.11/site-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/home/pushkar/ActionAI_2/nlp2llm/.venv-mix/lib/python3.11/site-packages/torchtext/data/__init

In this full code, `yield` is the bridge that allows the **`build_vocab_from_iterator`** function to do its job.

Here is exactly why `yield` is necessary for this specific script to work:

### 1. It satisfies the "Iterator" requirement

The function `build_vocab_from_iterator` is literally named "from iterator." It is designed to "consume" data one piece at a time. It doesn't want a giant list; it wants a **stream** of tokens.

When you pass `yield_token(sentences)` into it, the `vocab` builder starts a loop behind the scenes that looks like this:

* It asks your generator for tokens.
* Your generator **yields** `['i', 'love', 'my', 'dog']`.
* The vocab builder counts those words.
* It asks for more. Your generator **yields** `['i', 'love', 'my', 'cat']`.
* The vocab builder counts those.

### 2. It handles the "Duplicate" Logic

Look at your output for `vocab_list`. You’ll notice that "love" and "my" only appear **once**, even though they are in both sentences.

If you didn't use an iterator/generator approach with a large dataset, you would be storing millions of duplicate words in a list before even starting to build the vocabulary. By using `yield`, the code:

1. Grabs a sentence.
2. Updates the counts (e.g., "love" = 1).
3. Throws that sentence away from memory.
4. Grabs the next sentence.
5. Updates the counts (e.g., "love" = 2).

### 3. The Resulting Vocabulary

Because of that `yield` loop, the `vocab` object was able to build these two mappings:

* **`vocab.get_itos()` (Index to String):** A list where the position tells you the word.
* Example: `['love', 'my', 'cat', 'dog', 'i']` (Order may vary based on frequency).


* **`vocab.get_stoi()` (String to Index):** A dictionary where you look up a word to get its ID.
* Example: `{'love': 0, 'my': 1, ...}`



---

### Summary of the "Flow"

1. **Source:** `sentences` (The raw text).
2. **Pipe:** `yield_token` (The generator that tokenizes on-the-fly).
3. **Machine:** `build_vocab_from_iterator` (The tool that catches the yielded tokens and counts them).
4. **Product:** `vocab` (The final dictionary).

If you had used `return` and a list instead of `yield`, this code would still work for these 2 sentences, but if you tried to run it on a 5GB text file, your computer would likely run out of memory and the program would crash. **`yield` makes this code "production-ready."**



In the context of Natural Language Processing (NLP) and libraries like `torchtext`, you use `yield` primarily for **efficiency and scalability**.

While it might seem unnecessary for two short sentences, imagine you are training a model on the entire text of Wikipedia.

Here are the three main reasons why `yield` is used in that specific example:

### 1. Memory Efficiency (The "Water Pipe" vs. "Water Bucket")

If you have 10 million sentences and use a standard `return` function, Python has to store all 10 million tokenized lists in your RAM at the same time. This will likely crash your computer (MemoryError).

By using `yield`, you create a "stream." Only one sentence is being tokenized and stored in memory at any given second. As soon as the model "eats" those tokens, they are cleared out to make room for the next sentence.

### 2. Pipelining (Starting faster)

With a `return` function, you have to wait for the **entire** dataset to be tokenized before you can see a single result.
With `yield`, the moment the first sentence is processed, it is sent to the next step. This allows your training process to start immediately while the rest of the data is still being processed in the background.

### 3. Compatibility with PyTorch/Vocab

In the `torchtext` library, functions like `build_vocab_from_iterator` specifically **expect** an iterator (a generator).

The library is designed to "walk" through your data sentence by sentence to count how many times each word appears. If you passed a massive pre-tokenized list, it would be redundant and wasteful. Using `yield` allows `torchtext` to pull data from your file or list only as it needs it.

---

### Comparison of the two approaches:

**The Memory-Heavy Way (`return`):**

```python
def get_all_tokens(sentences):
    results = []
    for s in sentences:
        results.append(tokenizer(s))
    return results # Returns one GIANT list

```

**The ML-Standard Way (`yield`):**

```python
def yield_tokens(sentences):
    for s in sentences:
        yield tokenizer(s) # Returns one sentence at a time

```

### Summary

You use `yield` because NLP data is usually **Big Data**. It turns your function from a "Storage Tank" into a "Conveyor Belt."

Would you like to see how to pass this `yield_token` function into `build_vocab_from_iterator` to actually create a word dictionary?

In [5]:
print(vocab_list[1])
print(vocab["love"])

love
1


Lets add another word. If you add another sentence, you'll notice new words in the vocabulary and new punctuation is still ignored as expected.


### TensorFlow

In [6]:
import tensorflow as tf

# sample inputs

sentences = [
    "I love my dog",
    "i love my cat",
    "You love my dog!"]

# Initialize the layeer
vectorized_layer = tf.keras.layers.TextVectorization()

# Build the vocab
vectorized_layer.adapt(sentences)

# get vocab
vocab = vectorized_layer.get_vocabulary(include_special_tokens = False)

In [7]:
for index, word in enumerate(vocab):
    print(index, word)

0 my
1 love
2 i
3 dog
4 you
5 cat


### PyTorch

In [8]:
import torch
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer

sentences = [
    "I love my dog",
    "i love my cat",
    "You love my dog!"]

tokenizer = get_tokenizer("basic_english")

def yield_token(sentences):
    for sentence in sentences:
        yield tokenizer(sentence)

vocab = build_vocab_from_iterator(yield_token(sentences), specials=[])

vocab_list_1 = vocab.get_itos()
vocab_list_2 = vocab.get_stoi()

print(vocab_list_1)
print(vocab_list_2)

['love', 'my', 'dog', 'i', '!', 'cat', 'you']
{'cat': 5, '!': 4, 'i': 3, 'dog': 2, 'my': 1, 'you': 6, 'love': 0}



Now that you see how it behaves, let's include the two special tokens. The first one at 0 is used for padding and 1 is used for out-of-vocabulary words. These are important when you use the layer to convert input texts to integer sequences. You'll see that in the next lab.


### TensorFlow

In [18]:
import tensorflow as tf

# sample inputs

sentences = [
    "I love my dog",
    "i love my cat",
    "You love my dog!"]

# Initialize the layeer
vectorized_layer = tf.keras.layers.TextVectorization()

# Build the vocab
vectorized_layer.adapt(sentences)

# get vocab
vocab = vectorized_layer.get_vocabulary()

In [20]:
for index, words in enumerate(vocab):
    print(index, words)

0 
1 [UNK]
2 my
3 love
4 i
5 dog
6 you
7 cat


### PyTorch

In [ ]:
import torch
from torchtext.vocab import build_vocab_from_iterator
from torchtext.data.utils import get_tokenizer

sentences = [
    "I love my dog",
    "i love my cat",
    "You love my dog!"]

tokenizer = get_tokenizer("basic_english")

def yield_token(sentences):
    for sentence in sentences:
        yield tokenizer(sentence)

"""
Add <pad> and <unk> as specials; order matters: pad first (idx 0), unk second (idx 1)
Usually padding token is assigned index 0 because many PyTorch functions (like nn.Embedding) expect padding_idx=0.
"""

specials = ['<pad>', '<unk>']
vocab = build_vocab_from_iterator(yield_token(sentences), specials=specials)  #or specials=['<UNK>']

vocab_list_1 = vocab.get_itos()
# vocab_list_2 - vocab.get_stoi()

print(vocab_list_1)
# print(vocab_list_2)

['<pad>', '<unk>', 'love', 'my', 'dog', 'i', '!', 'cat', 'you']


Conclusion on vocab building!